# Agentic Search Quick Demo — OpenAI + Tavily

A simple Colab demo showing an agentic search loop:

**User → OpenAI → Tavily Search → OpenAI → More Search if needed → Final Answer**

Tavily replaces Brave as the external web-search tool. OpenAI remains the reasoning/orchestration layer.

## 1. Install dependencies


In [1]:
!pip -q install -U openai tavily-python


## 2. Add API keys

In Colab, add these as Secrets:

- `OPENAI_API_KEY`
- `TAVILY_API_KEY`

You can also set them with environment variables.

In [2]:
import os

try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
except Exception:
    pass

assert os.getenv("OPENAI_API_KEY"), "Missing OPENAI_API_KEY"
assert os.getenv("TAVILY_API_KEY"), "Missing TAVILY_API_KEY"

print("API keys loaded.")


API keys loaded.


## 3. Create the OpenAI + Tavily agent


In [3]:
import json
from openai import OpenAI
from tavily import TavilyClient

client = OpenAI(api_key=os.environ["OPENAI_API_KEY"])
tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

# Change this if you want to use another OpenAI model available to your account.
MODEL = os.getenv("OPENAI_MODEL", "gpt-6.2-sol")
MAX_SEARCH_ROUNDS = 3

tools = [{
    "type": "function",
    "name": "web_search",
    "description": "Search the live web using Tavily. Use this for current, uncertain, or factual information that may have changed.",
    "parameters": {
        "type": "object",
        "properties": {
            "query": {
                "type": "string",
                "description": "A concise web search query."
            }
        },
        "required": ["query"],
        "additionalProperties": False
    },
    "strict": True
}]

instructions = """
You are an agentic research assistant.
Decide when web search is necessary. For current or uncertain facts, use web_search.
After seeing search results, decide whether another search is required to answer accurately.
Use evidence from the search results and do not invent facts.
When you have enough information, provide a concise answer and include useful source URLs when available.
"""


## 4. Tavily search function


In [4]:
def web_search(query: str):
    result = tavily.search(
        query=query,
        search_depth="basic",
        max_results=5,
        include_answer=True,
        include_raw_content=False,
    )
    return result


## 5. Run the agentic search loop

Notice the important part: **OpenAI decides whether to call the search tool.** Tavily only executes the search.

In [5]:
def run_agent(user_question: str):
    conversation = [{"role": "user", "content": user_question}]

    for round_no in range(1, MAX_SEARCH_ROUNDS + 1):
        print(f"\n--- Agent round {round_no} ---")

        response = client.responses.create(
            model=MODEL,
            instructions=instructions,
            input=conversation,
            tools=tools,
        )

        conversation += response.output

        function_calls = [
            item for item in response.output
            if item.type == "function_call" and item.name == "web_search"
        ]

        if not function_calls:
            print("\nFINAL ANSWER:\n")
            print(response.output_text)
            return response.output_text

        for call in function_calls:
            args = json.loads(call.arguments)
            query = args["query"]
            print(f"Searching Tavily: {query}")

            results = web_search(query)

            conversation.append({
                "type": "function_call_output",
                "call_id": call.call_id,
                "output": json.dumps(results),
            })

    # Final answer pass with tools disabled after the search budget is exhausted.
    final_response = client.responses.create(
        model=MODEL,
        instructions=instructions + "\nDo not request more tools. Give the best evidence-based final answer now.",
        input=conversation,
    )

    print("\nFINAL ANSWER:\n")
    print(final_response.output_text)
    return final_response.output_text


## 6. Demo

This question intentionally requires multiple pieces of information, so the agent can demonstrate iterative search.

In [6]:
question = "What is the latest stable Python release, and which NumPy version first supported it?"
run_agent(question)



--- Agent round 1 ---
Searching Tavily: site:python.org downloads latest Python release August 2026 Python 3.14 stable release
Searching Tavily: site:numpy.org release notes first NumPy support Python 3.14 2.3.3 2.4

--- Agent round 2 ---
Searching Tavily: latest stable Python download python.org latest Python 3.15.2 3.14.10 November 2026
Searching Tavily: NumPy first support Python 3.15 wheels release notes NumPy 2.5.0 2.5.1 2.5.2 Python 3.15

--- Agent round 3 ---

FINAL ANSWER:

The latest stable release is [Python 3.15.2](https://www.python.org/downloads/), released on October 25, 2026.

[NumPy 2.5.2](https://numpy.org/devdocs/release/2.5.2-notes.html) was the first stable NumPy release to list support for Python 3.15. It shipped wheels for Python 3.15.0rc1, ahead of Python 3.15’s final release.


'The latest stable release is [Python 3.15.2](https://www.python.org/downloads/), released on October 25, 2026.\n\n[NumPy 2.5.2](https://numpy.org/devdocs/release/2.5.2-notes.html) was the first stable NumPy release to list support for Python 3.15. It shipped wheels for Python 3.15.0rc1, ahead of Python 3.15’s final release.'

## 7. Try your own question


In [7]:
run_agent("What are the latest important developments in AI agent security and sandboxing?")



--- Agent round 1 ---
Searching Tavily: latest AI agent security sandboxing 2026 July August September agent sandbox OpenAI Anthropic Google security agent isolation releases 2026
Searching Tavily: 2026 latest AI agent security standards OWASP agentic applications NIST AI agent standards initiative MCP security authorization 2026
Searching Tavily: 2026 AI agent sandboxing coding agents Kubernetes agent sandbox gVisor Firecracker Anthropic Claude Code OpenAI Codex 2026 recent announcements
Searching Tavily: AI agent security July August 2026 recent agent security developments sandbox prompt injection AI agents 2026 August

--- Agent round 2 ---
Searching Tavily: site:openai.com agents sdk sandbox execution April 2026 OpenAI site:openai.com OR site:anthropic.com managed agents sandbox April 2026
Searching Tavily: site:anthropic.com investigating three real-world incidents cybersecurity evaluations July 30 2026 site:anthropic.com OR site:aisi.gov.uk incident unsanctioned agent behaviour 

'**The biggest shift in late 2026: agent security is moving from “put it in a container” to enforcing two separate boundaries—what code can do, and what actions the agent is authorized to take.** A sandbox can protect the host and still leave an agent free to misuse a GitHub, email, or production API it was legitimately granted.\n\n### What’s new and why it matters\n\n- **Isolated execution is becoming standard infrastructure.** Google Cloud made [GKE Agent Sandbox generally available](https://cloud.google.com/blog/products/containers-kubernetes/bringing-you-agent-sandbox-on-gke-and-agent-substrate), with gVisor isolation and warm pools, and introduced the open-source Agent Substrate for suspending and resuming agent workloads. [Docker Sandboxes](https://www.docker.com/blog/ai-coding-agent-horror-stories-security-risks) take a different approach: microVMs with their own guest kernels. When comparing products, check the actual isolation boundary, what gets mounted, and what network acce

## Architecture

```text
User
  ↓
OpenAI model
  ↓
Should I search?
  ↓
Tavily web_search tool
  ↓
Search results
  ↓
OpenAI model
  ↓
More search needed?
  ├── Yes → Tavily → OpenAI
  └── No  → Final answer
```

This is the core pattern you can later extend for Bazai Agent Paradise with SQL, vector search, APIs, MCP tools, enterprise search, and other tools.